In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np

In [ ]:
import os

# # Set KaggleHub cache to a directory inside /content/
os.environ["KAGGLEHUB_CACHE"] = "/content/data"

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import glob

image_path = "/kaggle/input/q3-stage3-2026/dataset/images"
# Get all image paths

# Define mask path
mask_path = "/kaggle/input/q3-stage3-2026/dataset/masks"
# Get all masks paths



In [ ]:
# dataset

import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np
# Custom Dataset Class
class  Underwater(Dataset):
    def __init__(self, image_path,mask_path, transform=None, target_transform=None):
        self.image_path = image_path
        self.mask_path = mask_path


        self.transform = transform
        self.target_transform = target_transform
        self.imags = []
        self.mask = []




        images_paths = glob.glob(f"{self.image_path}/*.jpg")
        maskes_paths = glob.glob(f"{self.mask_path}/*.png")
        for i in  os.listdir(images_paths):

          self.imags.extend(i)

        for i in os.listdir(maskes_paths):

          self.mask.extend(i)









    def __len__(self):
        return len(self.metadata)
    def __getitem__(self, idx):
        img_path = self.imags[idx]
        mask_path =  self.mask[idx]

        image = Image.open(img_path).convert("RGB")# عشانها صورة
        mask = Image.open(mask_path).convert("L")   # Convert mask to grayscale (1 channel = binary segmentation)
# حتى لو باينري ممكن تخزن بالداتا سيت مو قاري سكيل
        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            mask = self.target_transform(mask)# حيضيف شانل وماراح نسوي له سكيوز
            #  in binaryTarget: [N, 1, H, W]


        # Replace mask values with remapped values
        mask = remap_mask(mask)# نرجعه ري مابد

        return image, mask       # In image classification datasets, we return image and label. Here, we return image and mask


In [ ]:
image_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Standard ImageNet normalization
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),  # Keep segmentation masks intact
    transforms.PILToTensor(),# لايغير القيم

])

In [ ]:
from torch.utils.data import DataLoader, random_split

dataset = Underwater(image_path=image_path, mask_path=mask_path,
                                         transform=image_transforms, target_transform=mask_transforms)
# 3. Split into Train and Test (e.g., 80/20 split)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

print(f"Total: {len(dataset)}, Train: {len(train_dataset)}, Test: {len(test_dataset)}")


# Create Train & Test DataLoaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)


In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# TO DO
import segmentation_models_pytorch as smp

# Define U-Net Model
device = "cpu"# or cuda
model = smp.Unet(
    encoder_name="efficientnet-b1",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=8,  #multi clas
).to(device)


In [ ]:
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm
# multiclass-cross enrtoby use
# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0


    for images, masks in tqdm(dataloader):# mask after ten[n,c,h,m] dim1=c
        images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)
        # mask shape becomes [N, H, W] نرجعه لشكله الحقيقي
        # لان يوم نحطه لتينسور في ترانفورم يضيف له شانيل مع انه هو ماله شانيل اصلا


        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

#  Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)    # mask shape becomes [N, H, W]

            outputs = model(images)    # [N, C, H, W]

            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# TO DO
import torch
from torch import nn


# Initialize model
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10  # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")


In [ ]:
import matplotlib.pyplot as plt

plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

In [ ]:
# TO DO